# Fare and Tip Patterns

**Grain:** pickup year/month × pickup borough × raw payment-type code. This product compares the recorded fare, total paid, and tip for each payment group. `payment_type` is kept as the source code; the notebook does not guess a label for unknown codes. `recorded_tip_pct_of_base_fare` is a ratio of sums for rows with positive base fare and nonnegative recorded tip, rather than an average of per-trip percentages. Cash tips may not be fully recorded, so this is a reporting metric, not a measure of every tip actually paid.


In [ ]:
from pathlib import Path
import sys

# Find the project even when Jupyter starts in its notebooks directory.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root / "notebooks" / "data-products"))
from product_utils import get_spark, register_integrated, publish_product

spark = get_spark()
register_integrated(spark)


## Build the product

Run this cell to replace the Delta snapshot. The SQL comments explain the key metric choices.


In [ ]:
query = """
WITH grouped AS (
    SELECT
        pickup_year,
        pickup_month,
        COALESCE(pickup_borough, 'Unknown') AS pickup_borough,
        payment_type,
        COUNT(trip_id) AS trip_count,
        COUNT(tip_amount) AS trips_with_recorded_tip_value,
        -- AVG ignores nulls; a recorded zero tip remains zero.
        ROUND(AVG(fare_amount), 2) AS avg_base_fare,
        ROUND(AVG(total_amount), 2) AS avg_total_amount,
        ROUND(AVG(tip_amount), 2) AS avg_recorded_tip_amount,
        SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) AS trips_with_positive_recorded_tip,
        -- Use the same eligible rows in numerator and denominator.
        SUM(CASE WHEN fare_amount > 0 AND tip_amount >= 0 THEN tip_amount END)
            AS eligible_recorded_tip_total,
        SUM(CASE WHEN fare_amount > 0 AND tip_amount >= 0 THEN fare_amount END)
            AS eligible_base_fare_total
    FROM integrated_taxi_trips
    WHERE trip_id IS NOT NULL
    GROUP BY pickup_year, pickup_month,
             COALESCE(pickup_borough, 'Unknown'), payment_type
)
SELECT
    pickup_year, pickup_month, pickup_borough, payment_type,
    trip_count, trips_with_recorded_tip_value,
    avg_base_fare, avg_total_amount, avg_recorded_tip_amount,
    trips_with_positive_recorded_tip,
    ROUND(100.0 * trips_with_positive_recorded_tip / NULLIF(trip_count, 0), 2)
        AS positive_recorded_tip_share_pct,
    ROUND(100.0 * eligible_recorded_tip_total
          / NULLIF(eligible_base_fare_total, 0), 2)
        AS recorded_tip_pct_of_base_fare
FROM grouped
"""
result = spark.sql(query)
product = publish_product(spark, result, "fare_and_tip_patterns")
product.printSchema()
product.show(20, truncate=False)
